## Deployment Sequence

# 1. One-time backend setup
bash scripts/bootstrap.sh

# 2. Fill in your two values
vim environments/prod/terraform.tfvars
# → aws_account_id, acm_certificate_arn

# 3. Standard Terraform flow (~20 min, MWAA is slow)
terraform init
terraform plan -var-file=environments/prod/terraform.tfvars
terraform apply -var-file=environments/prod/terraform.tfvars

# 4. Populate secrets + init pgvector + upload DAGs
bash scripts/post-apply.sh

Finishing the remaining files — staging environment, CloudWatch dashboards/alarms, and a `.terraform-version` pin.Now wire monitoring into `main.tf` and add the missing variable to `variables.tf`:Add the `alb_arn_suffix` output to the ALB module (monitoring needs it):Now let's verify the complete structure looks right:All cross-module references resolve cleanly. Let's copy everything to the output directory:Here's what was built and what to do with it.

---

## What's in the Terraform package

**9 modules, fully wired:**

| Module | Key decisions |
|--------|--------------|
| `vpc` | 3-AZ, private/public subnets, single NAT GW (add per-AZ for full HA), 7 VPC endpoints including `bedrock-runtime` (eliminates NAT egress for LLM calls) |
| `ecr` | Immutable tags, scan on push, lifecycle = keep 10 tagged / purge untagged after 1 day |
| `iam` | Least-privilege: ECS task role scoped to Bedrock Claude 3.5 Sonnet ARN only, your S3 prefix only, your Secrets Manager prefix only |
| `secrets` | Auto-generates DB password; all other secrets created as `REPLACE_ME` placeholders with `ignore_changes` so CI/CD can populate without Terraform overwriting |
| `rds` | PostgreSQL 16 (pgvector-capable), Multi-AZ, gp3 + auto-scaling storage, Performance Insights, 14-day backups, KMS-encrypted |
| `elasticache` | Redis 7 replication group, TLS + AUTH token (stored in Secrets Manager), LRU eviction, KMS-encrypted |
| `alb` | HTTPS with TLS 1.3 policy, HTTP→HTTPS redirect, WAF v2 with AWS managed OWASP + rate limit 1000 req/5min at edge, ALB access logs to S3 |
| `ecs` | All P0 env vars baked in (`ENABLE_MOCK=false`, `LLM_PROVIDER=bedrock`, `MEMORY_BACKEND=postgres`, `REDIS_TLS=true`); CPU+memory autoscaling on the API service; ECS Exec enabled for debugging |
| `mwaa` | Managed Airflow 2.9, private-only webserver, your existing DAGs upload to S3 unchanged, Airflow metadata DB pointed at RDS |
| `monitoring` | CloudWatch dashboard, alarms for ECS (CPU/mem/task count), RDS (CPU/connections/storage/lag), Redis (CPU/memory/evictions), ALB (5xx/latency/WAF blocks), log metric filters for ERROR lines + guardrail blocks + token budget hits |

## Deployment sequence

```bash
# 1. One-time backend setup
bash scripts/bootstrap.sh

# 2. Fill in your two values
vim environments/prod/terraform.tfvars
# → aws_account_id, acm_certificate_arn

# 3. Standard Terraform flow (~20 min, MWAA is slow)
terraform init
terraform plan -var-file=environments/prod/terraform.tfvars
terraform apply -var-file=environments/prod/terraform.tfvars

# 4. Populate secrets + init pgvector + upload DAGs
bash scripts/post-apply.sh
```

## One thing to do in your app code before deploying

The `REDIS_TLS=true` env var is set in ECS, but your `core/cache.py` likely connects without TLS. Add `ssl=True` to the Redis client init when that env var is set:

```python
import ssl, os
ssl_ctx = ssl.create_default_context() if os.getenv("REDIS_TLS") == "true" else None
client = Redis(host=host, port=port, password=auth_token, ssl=ssl_ctx is not None, ssl_cert_reqs=None)
```

The next natural step after this is the **CI/CD pipeline** — GitHub Actions that builds images, pushes to ECR, and triggers ECS rolling deployments automatically. Want to tackle that next?

import ssl, os
ssl_ctx = ssl.create_default_context() if os.getenv("REDIS_TLS") == "true" else None
client = Redis(host=host, port=port, password=auth_token, ssl=ssl_ctx is not None, ssl_cert_reqs=None)